In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest


In [ ]:
df = pd.read_excel("Причины пожаров.xlsx")

df.head()


In [ ]:
df.info()


In [ ]:
df = df.dropna(subset=["Полное описание"])
df["Полное описание"] = df["Полное описание"].str.lower()


In [ ]:
def classify_reason(text):
    if any(word in text for word in ["износ", "неисправн", "отказ", "коротк"]):
        return "Техническая"
    if any(word in text for word in ["наруш", "не соблюд", "регламент"]):
        return "Организационная"
    if any(word in text for word in ["персонал", "невним", "ошибка"]):
        return "Человеческий фактор"
    if any(word in text for word in ["погода", "внешн"]):
        return "Внешняя среда"
    return "Прочее"


In [ ]:
df["Категория причины"] = df["Полное описание"].apply(classify_reason)

df["Категория причины"].value_counts()



In [ ]:
pareto = df["Категория причины"].value_counts().reset_index()
pareto.columns = ["Причина", "Количество"]
pareto["Накопленный %"] = pareto["Количество"].cumsum() / pareto["Количество"].sum() * 100


In [ ]:
fig, ax1 = plt.subplots()

ax1.bar(pareto["Причина"], pareto["Количество"])
ax1.set_ylabel("Количество происшествий")

ax2 = ax1.twinx()
ax2.plot(pareto["Причина"], pareto["Накопленный %"], marker="o")
ax2.set_ylabel("Накопленный процент")

plt.title("Диаграмма Парето причин пожаров")
plt.show()


In [ ]:
df["Длина описания"] = df["Полное описание"].str.len()


In [ ]:
sns.boxplot(x=df["Длина описания"])
plt.title("Выявление аномалий методом ящика с усами")
plt.show()


In [ ]:
model = IsolationForest(contamination=0.05, random_state=42)
df["Аномалия"] = model.fit_predict(df[["Длина описания"]])



In [ ]:
df[df["Аномалия"] == -1].head()


In [ ]:
risk_map = {
    "Техническая": (3, 3),
    "Организационная": (2, 3),
    "Человеческий фактор": (2, 2),
    "Внешняя среда": (1, 3),
    "Прочее": (1, 1)
}

df["Вероятность"] = df["Категория причины"].apply(lambda x: risk_map[x][0])
df["Ущерб"] = df["Категория причины"].apply(lambda x: risk_map[x][1])
df["Риск"] = df["Вероятность"] * df["Ущерб"]


In [ ]:
risk_matrix = df.pivot_table(
    values="Риск",
    index="Вероятность",
    columns="Ущерб",
    aggfunc="mean"
)

sns.heatmap(risk_matrix, annot=True, cmap="Reds")
plt.title("Матрица рисков")
plt.show()


In [ ]:
example = df.iloc[0]["Полное описание"]
example


In [ ]:
swiss_cheese = pd.DataFrame({
    "Слой защиты": [
        "Проектирование",
        "Регламент технического обслуживания",
        "Контроль персонала",
        "Эксплуатация",
        "Аварийная защита"
    ],
    "Назначение слоя": [
        "Предотвращение конструктивных дефектов",
        "Своевременное выявление неисправностей",
        "Соблюдение инструкций",
        "Безопасная эксплуатация оборудования",
        "Минимизация последствий"
    ],
    "Отказ слоя": [
        "Конструктивный дефект не выявлен",
        "ТО проведено формально",
        "Контроль не выполнен",
        "Оборудование эксплуатировалось с дефектом",
        "Защита не сработала своевременно"
    ]
})

swiss_cheese



In [ ]:
swiss_cheese["Источник"] = [
    "Технический",
    "Организационный",
    "Человеческий фактор",
    "Человеческий фактор",
    "Технический"
]



In [ ]:
swiss_cheese["Пробой (1 — да, 0 — нет)"] = [1, 1, 1, 1, 1]
swiss_cheese
